# Lab Assignment 3 - Task 2

**Name:** Akshat  
**Roll number:** 12340160

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` - a non-terminal produces two non-terminals.
2. `A -> 'word'` - a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [16]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [17]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser
from nltk.parse.pchart import InsideChartParser

### 1. Converting the grammar to CNF

Every rule in the given grammar is already either `A -> B C` or `A -> 'word'`, except one:

```text
VP -> V NP PP [0.3]
```

This has three symbols on the right hand side, so it breaks CNF. It is replaced by a pair of
binary rules with a new intermediate non terminal `VP_PP`:

```text
VP    -> V VP_PP [0.3]
VP_PP -> NP PP   [1.0]
```

The probability is preserved because the two new rules are always used together, and they are the
only way to reach and leave `VP_PP`:

$$P(\text{VP} \to \text{V VP\_PP}) \times P(\text{VP\_PP} \to \text{NP PP}) = 0.3 \times 1.0 = 0.3$$

which is exactly the probability of the original `VP -> V NP PP` rule. The new non terminal also
keeps the grammar consistent, since the probabilities of all rules with the same left hand side
still add up to 1 (`VP`: 0.7 + 0.3, and `VP_PP`: 1.0).

In [18]:
cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

cnf_grammar = PCFG.fromstring(cnf_grammar_str)

print(cnf_grammar)
print("Is the grammar in CNF?", cnf_grammar.is_chomsky_normal_form())

Grammar with 14 productions (start state = S)
    S -> NP VP [1.0]
    NP -> 'I' [0.1]
    NP -> Det N [0.3]
    NP -> NP PP [0.6]
    VP -> V NP [0.7]
    VP -> V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8]
    Det -> 'a' [0.2]
    N -> 'man' [0.5]
    N -> 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
Is the grammar in CNF? True


### 2. Parsing with `ViterbiParser`

In [19]:
words = sentence.split()

viterbi_parser = ViterbiParser(cnf_grammar)
best_tree = list(viterbi_parser.parse(words))[0]

print("Most probable parse tree:")
print(best_tree)
best_tree.pretty_print()
print("Total probability:", best_tree.prob())

Most probable parse tree:
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|___                      
 |   |               NP                   
 |   |        _______|____                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with  a      telescope

Total probability: 0.0001512


### 3. Why this tree and not the other one

To compare the two readings without hard-coding anything, the rules of each tree are looked up in
the grammar and their probabilities are multiplied.

In [20]:
def rule_probabilities(tree, grammar):
    """Return the (rule, probability) pairs used in a tree, in the order they appear."""
    used = []
    for production in tree.productions():
        for rule in grammar.productions(lhs=production.lhs()):
            if rule.rhs() == production.rhs():
                used.append((production, rule.prob()))
    return used


def tree_probability(tree, grammar):
    """Product of the probabilities of every rule used in the tree."""
    total = 1.0
    for production, prob in rule_probabilities(tree, grammar):
        total *= prob
    return total


def show_tree_rules(tree, grammar, title):
    print(title)
    print("-" * 46)
    total = 1.0
    for production, prob in rule_probabilities(tree, grammar):
        total *= prob
        print(f"{str(production):<32}{prob:>8}")
    print("-" * 46)
    print(f"{'product':<32}{total:>12.6e}")
    print()
    return total

In [21]:
# InsideChartParser returns every parse, so the competing reading is taken from the parser
# itself instead of being written out by hand
inside_parser = InsideChartParser(cnf_grammar)
all_trees = sorted(inside_parser.parse(words), key=lambda t: t.prob(), reverse=True)

print("Number of parses found:", len(all_trees))
for tree in all_trees:
    print(f"\nprobability = {tree.prob():.6e}")
    print(tree)

Number of parses found: 2

probability = 1.512000e-04
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

probability = 1.080000e-04
(S
  (NP I)
  (VP
    (V saw)
    (VP_PP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.000108)


In [22]:
selected = all_trees[0]
competing = all_trees[1]

p_selected = show_tree_rules(selected, cnf_grammar, "Selected parse (PP attached to the NP)")
p_competing = show_tree_rules(competing, cnf_grammar, "Competing parse (PP attached to the VP)")

print(f"P(selected)  = {p_selected:.6e}")
print(f"P(competing) = {p_competing:.6e}")
print(f"ratio        = {p_selected / p_competing:.4f}")
print("\nSame as the probability reported by ViterbiParser?",
      abs(p_selected - best_tree.prob()) < 1e-12)

Selected parse (PP attached to the NP)
----------------------------------------------
S -> NP VP                           1.0
NP -> 'I'                            0.1
VP -> V NP                           0.7
V -> 'saw'                           1.0
NP -> NP PP                          0.6
NP -> Det N                          0.3
Det -> 'the'                         0.8
N -> 'man'                           0.5
PP -> P NP                           1.0
P -> 'with'                          1.0
NP -> Det N                          0.3
Det -> 'a'                           0.2
N -> 'telescope'                     0.5
----------------------------------------------
product                         1.512000e-04

Competing parse (PP attached to the VP)
----------------------------------------------
S -> NP VP                           1.0
NP -> 'I'                            0.1
VP -> V VP_PP                        0.3
V -> 'saw'                           1.0
VP_PP -> NP PP                       

In [23]:
# the two trees only differ in a few rules, so print just those
selected_rules = [str(p) for p in selected.productions()]
competing_rules = [str(p) for p in competing.productions()]

print("only in the selected parse :", [r for r in selected_rules if r not in competing_rules])
print("only in the competing parse:", [r for r in competing_rules if r not in selected_rules])

only in the selected parse : ['VP -> V NP', 'NP -> NP PP']
only in the competing parse: ['VP -> V VP_PP', 'VP_PP -> NP PP']


### Mathematical explanation

`ViterbiParser` does not use any notion of meaning. It computes, for every possible tree, the
product of the probabilities of the rules used in that tree, and returns the tree with the largest
product:

$$P(\text{tree}) = \prod_{r \in \text{tree}} P(r), \qquad
\hat{t} = \arg\max_{t} P(t)$$

**Selected parse.** The PP is attached to the noun phrase, so *the man with a telescope* is one NP.
The rules used are

$$
\begin{aligned}
P(t_1) =\;& P(S \to NP\ VP) \cdot P(NP \to \text{'I'}) \cdot P(VP \to V\ NP) \cdot P(V \to \text{'saw'}) \\
&\cdot P(NP \to NP\ PP) \cdot P(NP \to Det\ N) \cdot P(Det \to \text{'the'}) \cdot P(N \to \text{'man'}) \\
&\cdot P(PP \to P\ NP) \cdot P(P \to \text{'with'}) \cdot P(NP \to Det\ N) \cdot P(Det \to \text{'a'}) \cdot P(N \to \text{'telescope'}) \\
=\;& 1.0 \times 0.1 \times 0.7 \times 1.0 \times 0.6 \times 0.3 \times 0.8 \times 0.5 \times 1.0 \times 1.0 \times 0.3 \times 0.2 \times 0.5 \\
=\;& 1.512 \times 10^{-4}
\end{aligned}
$$

**Competing parse.** The PP is attached to the verb phrase through the CNF rules introduced above:

$$
\begin{aligned}
P(t_2) =\;& P(S \to NP\ VP) \cdot P(NP \to \text{'I'}) \cdot P(VP \to V\ VP\_PP) \cdot P(V \to \text{'saw'}) \cdot P(VP\_PP \to NP\ PP) \\
&\cdot P(NP \to Det\ N) \cdot P(Det \to \text{'the'}) \cdot P(N \to \text{'man'}) \\
&\cdot P(PP \to P\ NP) \cdot P(P \to \text{'with'}) \cdot P(NP \to Det\ N) \cdot P(Det \to \text{'a'}) \cdot P(N \to \text{'telescope'}) \\
=\;& 1.0 \times 0.1 \times 0.3 \times 1.0 \times 1.0 \times 0.3 \times 0.8 \times 0.5 \times 1.0 \times 1.0 \times 0.3 \times 0.2 \times 0.5 \\
=\;& 1.08 \times 10^{-4}
\end{aligned}
$$

**Where the difference comes from.** Both trees use exactly the same lexical rules and the same rules
for *the man*, *with* and *a telescope*, so all of those factors cancel. What is left is

$$
\frac{P(t_1)}{P(t_2)} = \frac{P(VP \to V\ NP) \cdot P(NP \to NP\ PP)}{P(VP \to V\ VP\_PP) \cdot P(VP\_PP \to NP\ PP)}
= \frac{0.7 \times 0.6}{0.3 \times 1.0} = \frac{0.42}{0.30} = 1.4
$$

So the noun attachment is 1.4 times more probable. The grammar gives `VP -> V NP` a much higher
probability (0.7) than `VP -> V NP PP` (0.3), and `NP -> NP PP` is the most likely NP rule (0.6),
so building the PP inside the noun phrase costs less probability mass than building it inside the
verb phrase. Since `ViterbiParser` maximises the product, it returns the reading *I saw the man who
was holding a telescope*.

Note that this is a property of the numbers in the grammar and not of the meaning of the sentence.
In general the verb attachment wins whenever

$$P(VP \to V\ NP\ PP) > P(VP \to V\ NP) \cdot P(NP \to NP\ PP)$$

Writing $p$ for the probability of `VP -> V NP PP`, the other `VP` rule must take $1 - p$, so the
condition becomes $p > 0.6(1 - p)$, that is $p > 0.375$. The grammar sets $p = 0.3$, which is below
that threshold, so the noun attachment wins.

# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**

In [24]:
# If needed, install spaCy and the English model first:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from collections import deque
from spacy import displacy

## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

In [25]:
nlp = spacy.load("en_core_web_sm")

sentence = ("The exhausted researchers at the institute discovered a new vaccine "
            "that completely prevents the viral mutation.")

parsed_doc = nlp(sentence)

print(parsed_doc)
print("number of tokens:", len(parsed_doc))

The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.
number of tokens: 17


## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.

In [26]:
def find_root(parsed_doc):
    """Return the token whose dependency label is ROOT."""
    for token in parsed_doc:
        if token.dep_ == "ROOT":
            return token
    return None


root_node = find_root(parsed_doc)
print("Root of the sentence:", root_node.text, "|", root_node.pos_)

Root of the sentence: discovered | VERB


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

In [27]:
def get_terminals(root_node, parsed_doc):
    """Return the subject of the root as the start node, and 'mutation' as the end node."""
    start_node = None
    for child in root_node.children:
        if child.dep_ == "nsubj":
            start_node = child
            break

    end_node = None
    for token in parsed_doc:
        if token.text == "mutation":
            end_node = token
            break

    return start_node, end_node


start_node, end_node = get_terminals(root_node, parsed_doc)
print("start node:", start_node.text, "| dependency:", start_node.dep_)
print("end node  :", end_node.text, "| dependency:", end_node.dep_)

start node: researchers | dependency: nsubj
end node  : mutation | dependency: dobj


## 4. Graph Traversal - Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

In [28]:
def bfs_shortest_path(start_node, end_node):
    """BFS over the dependency tree treated as an undirected graph."""
    queue = deque([[start_node]])
    visited = {start_node.i}

    while queue:
        path = queue.popleft()
        token = path[-1]

        if token == end_node:
            return path

        # neighbours are the children plus the head (the root is its own head)
        neighbours = list(token.children)
        if token.head != token:
            neighbours.append(token.head)

        for neighbour in neighbours:
            if neighbour.i not in visited:
                visited.add(neighbour.i)
                queue.append(path + [neighbour])

    return []


dependency_path = bfs_shortest_path(start_node, end_node)

print("Shortest path:", " -> ".join(token.text for token in dependency_path))
print("Number of tokens on the path:", len(dependency_path))

Shortest path: researchers -> discovered -> vaccine -> prevents -> mutation
Number of tokens on the path: 5


## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

In [29]:
def analyse_path(dependency_path):
    """Print the POS tag, head and dependency label of every token on the path."""
    print(f"| {'Word':<12} | {'POS':<6} | {'Head':<12} | {'Dependency':<10} |")
    print(f"| {'-' * 12} | {'-' * 6} | {'-' * 12} | {'-' * 10} |")
    for token in dependency_path:
        print(f"| {token.text:<12} | {token.pos_:<6} | {token.head.text:<12} | {token.dep_:<10} |")


analyse_path(dependency_path)

| Word         | POS    | Head         | Dependency |
| ------------ | ------ | ------------ | ---------- |
| researchers  | NOUN   | discovered   | nsubj      |
| discovered   | VERB   | discovered   | ROOT       |
| vaccine      | NOUN   | discovered   | dobj       |
| prevents     | VERB   | vaccine      | relcl      |
| mutation     | NOUN   | prevents     | dobj       |


The path is the syntactic skeleton of the sentence:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| discovered | VERB | discovered | ROOT |
| vaccine | NOUN | discovered | dobj |
| prevents | VERB | vaccine | relcl |
| mutation | NOUN | prevents | dobj |

Read from top to bottom it says: the researchers discovered a vaccine, and that vaccine prevents the
mutation. All the modifiers (`The`, `exhausted`, `at the institute`, `a`, `new`, `that`,
`completely`, `the`, `viral`) are left out because they hang off the path rather than lying on it.

## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.

In [30]:
displacy.render(parsed_doc, style="dep", jupyter=True, options={"distance": 110})

### Visual verification

Following the arrows in the diagram, the only way to get from *researchers* to *mutation* is

```text
researchers --nsubj--> discovered --dobj--> vaccine --relcl--> prevents --dobj--> mutation
```

Four edges, and the same five tokens printed by the table in step 5. Every other route is longer,
for example going through *at* and *institute* leads away from the root and reaches a dead end, so
BFS never expands it into a shorter path. The `nsubj` and `relcl` edges are traversed against the
direction of the arrow, which is allowed here because the tree is treated as undirected.